# Distillation: How Do You Compress a Large Model Into a Small One?

> **Background**: GPT-4 is strong but expensive and slow. Can we distill its capabilities into a 7B model so the 7B model approaches GPT-4 quality?
>
> Goal for this part: **understand three mainstream distillation methods for LLMs, and the end-to-end workflow of distilling from GPT-4 to a smaller model.**

One-sentence core idea:
**Distillation = a large model (Teacher) teaches a small model (Student). The student learns not only the final answers, but the teacher's "way of thinking".**

Note: for a deeper dive into On-Policy Distillation (OPD), see Part 23.


In [ ]:
import numpy as np

np.random.seed(42)

### 1. The essence: learn "how to judge" not only "what the answer is"

```
Standard SFT:
  Teacher output: "Paris"
  Student learns: input "Capital of France?" -> output "Paris"
  Problem: it learns the answer, but not the reasoning or uncertainty

Distillation:
  Teacher output: a probability distribution over tokens
    [Paris:0.90, London:0.05, Berlin:0.03, ...]
  Student learns: not only output "Paris", but match the whole distribution
  Benefit: Student learns teacher's calibration and preferences
```

Why is a distribution more valuable than a single answer?

"Paris 90%, London 5%, Berlin 3%" contains extra information:
1. London/Berlin are plausible but less likely ("dark knowledge")
2. many other options are near zero (explicitly telling the student what is wrong)

This is the key insight from Hinton's 2015 distillation work.


In [ ]:
# Interactive demo: hard labels vs soft labels (using a real probability distribution)
print("=== Hard labels vs soft labels ===")
print()

cities = ["Paris", "London", "Berlin", "Rome", "Madrid", "Tokyo", "Beijing", "Sydney"]
teacher_logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, -3.0, -4.0, -5.0])

# Hard labels (SFT): one-hot
hard_labels = np.zeros(len(cities))
hard_labels[0] = 1.0

print("Question: What is the capital of France?")
print()
print("Hard label (SFT):")
for city, prob in zip(cities[:5], hard_labels[:5]):
    bar = "#" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print('  -> The student only learns: "Paris is correct"')
print()

# Soft labels (distillation): a probability distribution
temperature = 3.0
scaled_logits = teacher_logits / temperature
soft_labels = np.exp(scaled_logits) / np.exp(scaled_logits).sum()

print("Soft label (distillation, T=3):")
for city, prob in zip(cities, soft_labels):
    bar = "#" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print("  -> The student also learns:")
print("     1. Paris is the most likely answer")
print("     2. London/Berlin are also European capitals (similarity signal)")
print("     3. Tokyo/Beijing are ~0 (unrelated)")
print()

# Quantify the information difference (entropy)
hard_entropy = -np.sum(hard_labels * np.log(hard_labels + 1e-10))
soft_entropy = -np.sum(soft_labels * np.log(soft_labels + 1e-10))
print(f"Hard-label entropy: {hard_entropy:.2f} bits")
print(f"Soft-label entropy: {soft_entropy:.2f} bits")
print(f"-> Soft labels carry ~{soft_entropy:.1f} bits of information, far more than hard labels ({hard_entropy:.2f} bits).")


### 2. Method 1: logit distillation (the classic)

Make the student's output distribution match the teacher's output distribution.

Loss:

$$L = (1-lpha) \cdot L_{CE}(S, y) + lpha \cdot T^2 \cdot L_{KL}(S_T, T_T)$$

Where:
- $L_{CE}$: cross entropy against the ground truth (keeps correctness)
- $L_{KL}$: KL divergence between student and teacher distributions (learns dark knowledge)
- $T$: temperature (larger T makes the distribution softer)
- $lpha$: balance weight

Temperature intuition:

```
T=1:  [0.90, 0.05, 0.03, 0.02]  (very sharp)
T=5:  [0.40, 0.25, 0.20, 0.15]  (softer; dark knowledge shows)
T=20: [0.28, 0.26, 0.24, 0.22]  (too soft; close to uniform)
```

Common range: T=3~10.


In [ ]:
# How temperature T changes the soft-label distribution
print("=== How temperature T affects soft labels ===")
print()

logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, 0.01, 0.001, 0.0001])
labels = ["Paris", "London", "Berlin", "Rome", "Madrid", "Vienna", "Prague", "Warsaw"]

for T in [1, 3, 10, 20]:
    scaled = logits / T
    probs = np.exp(scaled) / np.exp(scaled).sum()

    print(f"T={T:2d}: ", end="")
    for i in range(5):
        bar = "#" * int(probs[i] * 50)
        print(f"{labels[i]}:{probs[i]:.3f} {bar}  ", end="")
    print()

print()
print("T=1:  almost only Paris -> weaker signals are hidden")
print("T=3:  London/Berlin get some probability -> weaker signals show up")
print("T=10: more uniform -> richer signals but weaker peak")
print("T=20: nearly uniform -> too little useful signal")


### 3. Method 2: data distillation (the most practical)

Logit distillation assumes teacher and student share the **same tokenizer/vocab**.
For LLMs this is often false (GPT-4 vs Qwen, etc.).

**Data distillation bypasses this**: let the teacher generate high-quality training data; then do SFT on the student.

```
Step 1: collect prompts (from your real business use cases)
  ["Write a poem about spring", "Explain quantum mechanics", "Translate: Hello -> Chinese", ...]

Step 2: teacher generates answers
  GPT-4: "Spring is coming..."
  GPT-4: "Quantum mechanics studies..."

Step 3: train student on (prompt, teacher_answer)
  Standard SFT: student imitates teacher's style and quality
```

Pros: no vocab alignment requirement.
Cons: learns "what answers look like", but not the teacher's full distribution.

Advanced tricks:
- multi-turn dialogue distillation
- CoT distillation (teacher provides reasoning traces)
- rejection sampling (generate multiple answers, keep the best)


In [ ]:
# Simulate a tiny data-distillation workflow
print("=== Simulated data distillation workflow ===")
print()

prompts = [
    "Explain what machine learning is.",
    "Write a short poem about autumn.",
    "Explain the difference between list and tuple in Python.",
]

# Simulated Teacher (e.g., GPT-4) answers
teacher_responses = [
    "Machine learning is a branch of AI where a model learns patterns from data instead of being explicitly programmed.",
    "Autumn wind sweeps fallen leaves; frost arrives and flowers fade. Alone by a cold window, I think of you and wonder if your coat is warm.",
    "A list is mutable (you can add/remove/modify items), while a tuple is immutable (it cannot be changed after creation). Lists use [], tuples use ().",
]

print("Generated training samples:")
for i, (prompt, response) in enumerate(zip(prompts, teacher_responses)):
    print()
    print(f"--- Sample {i+1} ---")
    print(f"User: {prompt}")
    print(f"Assistant: {response}")

print()
print(f"Total samples: {len(prompts)}")
print("The student runs SFT on these samples to imitate the teacher's style.")
print()
print("In real projects, you often need 10k to 100k+ such samples.")


### 4. Method 3: feature distillation (advanced)

Not only match output distributions, but also match intermediate representations.

```
Teacher (96 layers):  ... -> Layer 48 -> ... -> Output
                           ^
Student (32 layers):  ... -> Layer 16 -> ... -> Output
  match student Layer 16 to teacher Layer 48 (after projection)
```

Why it can help: intermediate layers encode rich "understanding" signals.

Why it is less common for LLMs:
- closed-source teachers do not expose hidden states
- teacher/student dimensions differ (need projection)
- higher compute and memory

In practice, data distillation is the dominant approach in LLM distillation.


### 5. Practical recipe: distill a 7B model from GPT-4

Below is a concrete end-to-end workflow.


In [ ]:
print("=== Practical: distill GPT-4 -> 7B (workflow) ===")
print()

steps = [
    ("Step 1: Choose a base model", [
        "Examples: Qwen2.5-7B / Llama-3-8B / Mistral-7B",
        "Requirement: the base model should already be decent (not too weak)",
        "Prefer an instruct-tuned checkpoint (already follows instructions)",
    ]),
    ("Step 2: Collect prompts", [
        "Source 1: your business data (real user questions)",
        "Source 2: open datasets (OpenHermes, ShareGPT, WildChat)",
        "Source 3: synthetic prompts generated by another LLM",
        "Scale: at least 5k; commonly 50k to 100k",
    ]),
    ("Step 3: Teacher generates answers", [
        "Use a teacher model (e.g., GPT-4) to answer each prompt",
        "Example system prompt: 'You are a helpful assistant. Answer accurately and in detail.'",
        "temperature=0.7 (keeps some diversity)",
        "Rough cost example: 50k * ~500 tokens ~= 25M tokens",
    ]),
    ("Step 4: Clean the data", [
        "Drop answers that are too short (<20 tokens)",
        "Drop refusal-style answers (e.g., 'As an AI...')",
        "Fix broken formatting",
        "Deduplicate (e.g., keep only one when similarity > 0.9)",
    ]),
    ("Step 5: Run SFT", [
        "Tooling: LLaMA-Factory / Axolotl / Firefly",
        "Format: ChatML or ShareGPT",
        "Typical hyperparams: lr=2e-5, epochs=3, batch_size=128",
        "Hardware example: 4x A100 80GB, ~6-12 hours",
    ]),
    ("Step 6: Evaluate", [
        "Run standard benchmarks with lm-eval",
        "Do human eval on ~100 business samples",
        "Compare student vs teacher on your target tasks",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()


### 6. Distillation vs OPD: when to use which?

| Dimension | Data distillation | OPD (On-Policy Distillation) |
|:---|:---|:---|
| When teacher is used | before training (generate data) | during training (online scoring) |
| Student data | teacher-written answers | student-generated answers |
| Engineering complexity | low (basically SFT) | high (rollouts + online teacher) |
| Exposure bias | yes | no |
| Vocab requirement | none | often needs alignment (or cross-tokenizer distill) |
| Cost | lower (teacher runs once) | higher (teacher runs repeatedly) |
| Best for | fast iteration, limited budget | max performance with strong eng support |

Recommendation: start with data distillation to get a strong baseline quickly.
If it is not enough, consider OPD.


In [ ]:
# Simulated distillation outcome (toy numbers)
print("=== Distillation outcome comparison (simulated) ===")
print()

print("Assume the teacher is GPT-4 and scores 86.4 on MMLU.")
print()

models = [
    ("GPT-4 (Teacher)", 86.4, "baseline"),
    ("7B base (no distillation)", 64.2, "raw capability"),
    ("7B + data distillation", 72.8, "+8.6"),
    ("7B + data distillation + CoT", 75.1, "+10.9"),
    ("7B + OPD", 77.3, "+13.1 (but ~10x cost)"),
]

print(f"{'Model':<25s} {'MMLU':>8s} {'Gain':>10s}")
print("-" * 45)
for name, score, note in models:
    gain = score - 64.2
    print(f"{name:<25s} {score:>6.1f}  {gain:>+6.1f}  ({note})")

print()
print("Conclusion: data distillation is often the best ROI; OPD can be stronger but costs more.")
print("In practice, 'data distillation + CoT distillation' is a common combo.")


### 7. Common questions

| Question | Answer |
|:---|:---|
| Can student surpass teacher? | In theory no (upper bound is the teacher), but with extra data the student can outperform on some narrow tasks. |
| What is lost? | creativity, long-tail knowledge, complex reasoning are hardest to distill. |
| How much data? | minimum ~5k, recommended 50k+. Quality > quantity. |
| Different tokenizers? | use data distillation (no vocab alignment). |
| Need RLHF after distill? | depends; if teacher is aligned, student often inherits alignment. |
| Multi-teacher distillation? | yes: e.g., one teacher for reasoning, one for writing, one for multimodal. |


---

## Distillation Summary

1. [ok] Essence: learn teacher distributions (dark knowledge), not only hard labels
2. [ok] Logit distillation: match distributions; usually needs aligned vocab/tokenizer
3. [ok] Data distillation: teacher generates data; student does SFT (most practical)
4. [ok] Feature distillation: match hidden states; strong but expensive and often unavailable
5. [ok] Recipe: base model -> collect prompts -> teacher generate -> clean -> SFT -> eval
6. [ok] Distill vs OPD: data distill is cheaper; OPD is stronger but expensive
7. [ok] CoT distillation: teacher provides reasoning traces; student learns reasoning

One-sentence summary: distillation is a big teacher and a small student; the most practical path is to let GPT-4 write tens of thousands of answers and train the small model to imitate.
